In [27]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

presto_port = '80'
# username = 'airflow-user'
username = 'jatin.rajani1@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


In [63]:
## For Blr Exp Please Use First Query

In [62]:


city_name = 'Bangalore'
start_date='2024-04-23'
end_date='2024-05-06'
# service_name='CabEconomy'

In [52]:
def route_check_func(city_name,start_date,end_date,service_name):
    route_check=f"""WITH service_mapping AS ( 
            SELECT 
                service_detail_id,
                service_level as service_name,
                service_category,
                service_id,
                city_display_name as city,
                city_id
            FROM 
                datasets.service_mapping
            WHERE 
                city_display_name='{city_name}'
                and service_level='{service_name}'
        ),
        
        fe_tbl as (
                SELECT
                    fare_estimate_id fid,
                    api_context,final_amount as final_amount_fare,
                    date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                    cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                    epoch fe,ride_distance as d1, estimated_ride_time as t1
                FROM
                    pricing.fare_estimates_enriched 
                WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                   
        ),
        orderlogs as 
        (SELECT *,u.id_array as fare_estimate_id
        FROM 
        ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
                 date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                 epoch,estimate_id,estimate_ids,ride_time as t3,
                 fare_recalculated_reason, all_paths,
                 spd_fraud_flag,order_status,
                 service_obj_service_name as service_name,
                 fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
                 cast(json_parse(estimate_ids) as array<varchar>) AS id_array
          FROM orders.order_logs_snapshot
               WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND city_name IN (SELECT city FROM service_mapping)
                AND service_obj_service_name IN (SELECT service_name FROM service_mapping)) as t
        CROSS JOIN UNNEST(t.id_array) AS u(id_array)),
        immutable AS (
            SELECT 
                DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                order_id,
                cast (event_time as double) event_time
            FROM 
                orders.order_logs_immutable 
               WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_type = 'started'
                AND city_name IN (SELECT city FROM service_mapping)
                AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
        ),
        
        nav_metrics AS (
            SELECT 
                event_props_order_id,
                case when device_info_app_version>='5.3.22' then 'Above 5.3.22' else 'Below 5.3.22' end as app_version_type,
                case when device_info_app_version>='5.3.24' then 'Above 5.3.24' else 'Below 5.3.24' end as app_optimal_version_type,
                cast  (event_props_epoch as double) event_props_epoch,
                DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                try(cast(event_props_distance_in_meters as double) )/ 1000 d2
            FROM
                clevertap.captain_inappnavmetrics_immutable
                 WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_props_order_status = 'started'
        ),
        
        distance2 AS (
            SELECT 
                orderdate,
                order_id,
                app_version_type,
                app_optimal_version_type,
                d2
            FROM 
                ( 
                select 
                    imu.orderdate,
                    imu.order_id,
                    app_version_type,
                     app_optimal_version_type,
                    date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                    "d2"
                from 
                    immutable imu 
                join 
                    nav_metrics nav 
                    ON imu.order_id = nav.event_props_order_id
                )
                --and event_props_epoch >= event_time )
            WHERE abs(minDiff) <= 1
        ),
        done as  (select 
                   distinct order_id,d1,t1,t3,epoch,fare_recalculated_reason,service_name,rn,api_context, (fe_amount-order_amount) as amt_diff, all_paths,fid,
                spd_fraud_flag,order_status,orderdate
               
                   from 
                  (SELECT *,row_number () over(partition by order_id order by fe) rn from 
                  (SELECT  *
                   from orderlogs join fe_tbl
                   on fare_estimate_id = fid 
                   and service_detail_id = service_id_fare)
                   )
                   where rn =1
         ),
        dthree as (select distinct order_id, CASE
                WHEN selectedroute = 'hFRoute' THEN hfdistance
                WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
                WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
                ELSE NULL -- Handle other cases if needed
            END AS d3, selectedroute
            from
        (select * from (
        select *, row_number () over(partition by order_id order by updatedAt) rn from (
         select yyyymmdd,orderid as order_id, cast(hfdistance as double)*1000 as hfdistance,cast(pingdensity as double)*100 as pingdensity,cast(directionsresponsedistance as double) as directionsresponsedistance,
                     cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,selectedroute,updatedAt
                     from raw.captain_distance_response
                        WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                       
                    ))
                    where rn = 1
                     )),
                     
        merged_tbl as
        (
        
        
            select done.order_id,d2,done.orderdate,d3,d1,app_version_type,app_optimal_version_type,
            fare_recalculated_reason,spd_fraud_flag,order_status,service_name,
            CASE
                WHEN d2 IS NOT NULL then d1-d2
                END AS fe_vs_nav,
                d1-(d3*1.00/1000.00) as Est_Nav,
                case WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00)
                END AS nav_actual
            from done
            left join dthree
            on done.order_id=dthree.order_id
            left join distance2
            on done.order_id=distance2.order_id
        )
        
        
        select service_name,
        orderdate,
        CASE WHEN orderdate >= ('2024-04-23') and (orderdate<='2024-04-28') then 'pre' 
        when orderdate>=('2024-05-03') THEN 'post' end as new_date_type,
        app_version_type,
        count(distinct order_id) as gross_orders,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end) as net_orders,
        (count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end)*100.00)/(count (distinct order_id)) as "G2N",
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
        (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as fe_nav_bucket_within_500m_percent,
        (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as nav_actual_bucket_within_500m_percent,
        
        
        
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_bucket_within_500m,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='b. Less_than_500m' then order_id end) as fe_nav_bucket_less_500m,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='c. Greater_than_500m' then order_id end) as fe_nav_bucket_grt_500m,
        
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end) as nav_actual_bucket_within_500m,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='b. Less_than_500m' then order_id end) as nav_actual_bucket_less_500m,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='c. Greater_than_500m' then order_id end) as nav_actual_bucket_grt_500m,
        
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='a. Within_500_m' then order_id end) as Est_Nav_bucket_within_500m,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='b. Less_than_500m' then order_id end) as Est_Nav_bucket_less_500m,
        count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='c. Greater_than_500m' then order_id end) as Est_Nav_bucket_grt_500m
        
        
        
        from(
                select *,case when d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
                when d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
                when d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket,
                case when  d2 is not NULL and nav_actual>=-0.5 and nav_actual<=0.5 then 'a. Within_500_m'
                when d2 is not NULL and nav_actual<-0.5 then 'b. Less_than_500m'
                when d2 is not NULL and nav_actual>0.5 then 'c. Greater_than_500m' end as nav_actual_bucket,
                case when Est_Nav>=-0.5 and Est_Nav<=0.5 then 'a. Within_500_m'
                when Est_Nav<-0.5 then 'b. Less_than_500m'
                when Est_Nav>0.5 then 'c. Greater_than_500m' end as Est_Nav_bucket
                from merged_tbl
                group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14
        )
        group by 1,2,3,4
        order by 2
"""
    route_check_df=pd.read_sql(route_check,connection4)
    return route_check_df

In [53]:
total_df_d1_d2=pd.DataFrame()
city_name=['Bangalore','Delhi','Hyderabad','Chennai','Jaipur']
service_level=['Link','Auto','CabEconomy']
for i in city_name[0:1]:
    print(i)
    for j in service_level[1:]:
        print(j)
        df=route_check_func(i,start_date,end_date,j)
        # df['distance_type']='second_longest'
        df['fe_nav_bucket_within_500m_percent']=(df['fe_nav_bucket_within_500m']*100.00)/df['net_orders']
        df['nav_actual_bucket_within_500m_percent']=(df['nav_actual_bucket_within_500m']*100.00)/df['net_orders']
        df['route_diff_percent']=(df['route_diff_orders']*100.00)/df['net_orders']
        total_df_d1_d2=pd.concat([total_df_d1_d2,df])

Bangalore
Auto
CabEconomy


In [47]:
total_df_d1_d2

,service_name,orderdate,new_date_type,app_optimal_version_type,gross_orders,net_orders,G2N,route_diff_orders,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,route_diff_percent,Est_Nav_bucket_less_500m_percent,Est_Nav_bucket_grt_500m_percent
0,Auto,2024-04-23,pre,Above 5.3.24,411,410,99.76,101,89.024390,70.000000,365,27,73,287,79,98,295,33,82,24.634146,8.048780,20.000000
1,Auto,2024-04-23,pre,None,199897,3476,1.74,808,0.000000,0.000000,0,0,0,0,0,0,2546,251,679,23.245109,7.220944,19.533947
2,Auto,2024-04-23,pre,Below 5.3.24,135169,134752,99.69,32173,87.851757,73.322103,118382,10344,15033,98803,16261,28770,97906,10864,25974,23.875712,8.062218,19.275410
3,Auto,2024-04-24,pre,Below 5.3.24,138796,138372,99.69,29185,88.099471,76.170034,121905,11130,14301,105398,18477,23435,105015,13421,19920,21.091695,9.699217,14.395976
4,Auto,2024-04-24,pre,None,200184,3361,1.68,687,0.000000,0.000000,0,0,0,0,0,0,2570,302,489,20.440345,8.985421,14.549241
5,Auto,2024-04-24,pre,Above 5.3.24,445,442,99.33,77,88.461538,79.864253,391,31,69,353,72,69,353,33,56,17.420814,7.466063,12.669683
6,Auto,2024-04-25,pre,None,264669,3410,1.29,730,0.000000,0.000000,0,0,0,0,0,0,2574,316,520,21.407625,9.266862,15.249267
7,Auto,2024-04-25,pre,Below 5.3.24,152637,152172,99.70,32354,87.648188,75.661094,133376,12985,16239,115135,20008,27484,115135,14143,22880,21.261467,9.294088,15.035618
8,Auto,2024-04-25,pre,Above 5.3.24,447,443,99.11,88,90.519187,78.329571,401,23,68,347,72,69,346,32,65,19.864560,7.223476,14.672686
9,Auto,2024-04-26,pre,Below 5.3.24,112088,111744,99.69,23359,88.676797,76.257338,99091,9076,9685,85213,13360,19271,85768,10433,15537,20.904031,9.336519,13.904102


In [ ]:
total_df_d1_d2['service_name'].nunique()

In [37]:
total_df_d1_d2['Est_Nav_bucket_less_500m_percent']=(total_df_d1_d2['Est_Nav_bucket_less_500m']*100.00)/(total_df_d1_d2['net_orders'])
total_df_d1_d2['Est_Nav_bucket_grt_500m_percent']=(total_df_d1_d2['Est_Nav_bucket_grt_500m']*100.00)/(total_df_d1_d2['net_orders'])

In [42]:
total_df_d1_d2.columns

Index(['service_name', 'orderdate', 'new_date_type',
       'app_optimal_version_type', 'gross_orders', 'net_orders', 'G2N',
       'route_diff_orders', 'fe_nav_bucket_within_500m_percent',
       'nav_actual_bucket_within_500m_percent', 'fe_nav_bucket_within_500m',
       'fe_nav_bucket_less_500m', 'fe_nav_bucket_grt_500m',
       'nav_actual_bucket_within_500m', 'nav_actual_bucket_less_500m',
       'nav_actual_bucket_grt_500m', 'Est_Nav_bucket_within_500m',
       'Est_Nav_bucket_less_500m', 'Est_Nav_bucket_grt_500m',
       'route_diff_percent', 'Est_Nav_bucket_less_500m_percent',
       'Est_Nav_bucket_grt_500m_percent'],
      dtype='object')

In [56]:
k1=total_df_d1_d2.groupby(['service_name', 'new_date_type', 'app_version_type']).agg({
    'gross_orders': 'sum',
    'net_orders': 'sum',
    'route_diff_orders': 'sum',
    'fe_nav_bucket_within_500m': 'sum',
    'nav_actual_bucket_within_500m': 'sum',
    'Est_Nav_bucket_within_500m': 'sum',
    'Est_Nav_bucket_less_500m': 'sum',
    'Est_Nav_bucket_grt_500m': 'sum'
}).reset_index()


In [57]:
k1['Est_Nav_bucket_less_500m_percent']=(k1['Est_Nav_bucket_less_500m']*100.00)/(k1['net_orders'])
k1['Est_Nav_bucket_grt_500m_percent']=(k1['Est_Nav_bucket_grt_500m']*100.00)/(k1['net_orders'])
k1['fe_nav_bucket_within_500m_percent']=(k1['fe_nav_bucket_within_500m']*100.00)/k1['net_orders']
k1['nav_actual_bucket_within_500m_percent']=(k1['nav_actual_bucket_within_500m']*100.00)/k1['net_orders']
k1['route_diff_percent']=(k1['route_diff_orders']*100.00)/k1['net_orders']

In [59]:
k1

,service_name,new_date_type,app_version_type,gross_orders,net_orders,route_diff_orders,fe_nav_bucket_within_500m,nav_actual_bucket_within_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,Est_Nav_bucket_less_500m_percent,Est_Nav_bucket_grt_500m_percent,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,route_diff_percent
0,Auto,post,Above 5.3.22,426782,425293,88052,392133,328733,324650,31836,68161,7.485663,16.026833,92.203022,77.295653,20.703844
1,Auto,post,Below 5.3.22,16980,16905,3832,14156,12833,12523,1207,3161,7.139899,18.698610,83.738539,75.912452,22.667850
2,Auto,pre,Above 5.3.22,385246,384032,81811,339235,292063,292000,35608,56388,9.272144,14.683151,88.335087,76.051735,21.303173
3,Auto,pre,Below 5.3.22,421086,419772,90882,369638,317026,316730,38421,64596,9.152826,15.388354,88.056850,75.523379,21.650324
4,CabEconomy,post,Above 5.3.22,71854,71766,20815,63963,51636,49384,10378,11379,14.460887,15.855698,89.127163,71.950506,29.003985
5,CabEconomy,post,Below 5.3.22,1836,1832,666,1369,1221,1123,269,420,14.683406,22.925764,74.727074,66.648472,36.353712
6,CabEconomy,pre,Above 5.3.22,61711,61635,19495,51907,42745,40628,11441,9560,18.562505,15.510668,84.216760,69.351829,31.629756
7,CabEconomy,pre,Below 5.3.22,63004,62935,20712,52883,43114,40784,11876,10273,18.870263,16.323191,84.027965,68.505601,32.910145


In [60]:
total_df_d1_d2.to_csv('d1_new_v1.csv')

In [25]:
total_df_d1_d2.to_csv('d1_d2_app_version_v2.csv')

In [ ]:
total_df_d1_d2[['orderdate','app_version_type','net_orders','fe_nav_bucket_within_500m_percent']].sort_values(by='orderdate',ascending=True)

In [4]:
route_check_df=pd.read_sql(route_check,connection3)

In [5]:
route_check_df

,service_name,orderdate,gross_orders,net_orders,G2N,route_diff_orders,fe_nav_bucket_within_500m_percent,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
0,CabEconomy,2024-04-25,52496,23744,45.23,7560,36.60,19215,2347,3244,15904,5022,3756,15591,4426,3725
1,CabEconomy,2024-04-26,43344,19073,44.00,5953,36.18,15681,1809,2112,12858,3771,2906,12611,3547,2914
2,CabEconomy,2024-04-27,47075,22144,47.04,6952,38.89,18306,1933,2406,14940,4451,3169,14723,4196,3223
3,CabEconomy,2024-04-28,53933,21930,40.66,6989,33.01,17801,2150,2265,14681,4190,3265,14384,4182,3362
4,CabEconomy,2024-04-29,44584,21774,48.84,6489,40.48,18048,2093,1879,14815,3969,3183,14670,4059,3044
5,CabEconomy,2024-04-30,42184,22176,52.57,6599,43.20,18225,2446,1693,15216,3646,3493,15073,4028,3073
6,CabEconomy,2024-05-01,44411,22847,51.44,6895,42.42,18840,2539,1629,15696,3824,3447,15392,4394,3060
7,CabEconomy,2024-05-02,51897,23737,45.74,7200,37.66,19545,2382,1999,16361,3808,3706,16036,4117,3581
8,CabEconomy,2024-05-03,46317,23869,51.53,6644,44.47,20595,1885,1603,16948,3244,3865,16774,3347,3737
9,CabEconomy,2024-05-04,54846,25387,46.29,7556,39.73,21793,2030,1736,17678,3583,4268,17438,3769,4173


In [6]:
city_name = 'Bangalore'
start_date='2024-04-25'
end_date='2024-05-05'
service_name='Auto'

In [10]:
route_check1=f"""WITH service_mapping AS ( 
        SELECT 
            service_detail_id,
            service_level as service_name,
            service_category,
            service_id,
            city_display_name as city,
            city_id
        FROM 
            datasets.service_mapping
        WHERE 
            city_display_name='{city_name}'
            and service_level='{service_name}'
    ),
    
    fe_tbl as (
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1
            FROM
                pricing.fare_estimates_enriched 
            WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
               
    ),
    orderlogs as 
    (SELECT *,u.id_array as fare_estimate_id
    FROM 
    ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
             date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
             epoch,estimate_id,estimate_ids,ride_time as t3,
             fare_recalculated_reason, all_paths,
             spd_fraud_flag,order_status,
             service_obj_service_name as service_name,
             fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
             cast(json_parse(estimate_ids) as array<varchar>) AS id_array
      FROM orders.order_logs_snapshot
           WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)) as t
    CROSS JOIN UNNEST(t.id_array) AS u(id_array)),
    immutable AS (
        SELECT 
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
            order_id,
            cast (event_time as double) event_time
        FROM 
            orders.order_logs_immutable 
           WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_type = 'started'
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
    ),
    
    nav_metrics AS (
        SELECT 
            event_props_order_id,
            cast  (event_props_epoch as double) event_props_epoch,
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
            try(cast(event_props_distance_in_meters as double) )/ 1000 d2
        FROM
            clevertap.captain_inappnavmetrics_immutable
             WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_props_order_status = 'started'
    ),
    
    distance2 AS (
        SELECT 
            orderdate,
            order_id,
            d2
        FROM 
            ( 
            select 
                imu.orderdate,
                imu.order_id,
                date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                "d2"
            from 
                immutable imu 
            join 
                nav_metrics nav 
                ON imu.order_id = nav.event_props_order_id
            )
            --and event_props_epoch >= event_time )
        WHERE abs(minDiff) <= 1
    ),
    done as  (select 
               distinct order_id,d1,t1,t3,epoch,fare_recalculated_reason,service_name,rn,api_context, (fe_amount-order_amount) as amt_diff, all_paths,fid,
            spd_fraud_flag,order_status,orderdate
           
               from 
              (SELECT *,row_number () over(partition by order_id order by fe) rn from 
              (SELECT  *
               from orderlogs join fe_tbl
               on fare_estimate_id = fid 
               and service_detail_id = service_id_fare)
               )
               where rn =1
     ),
    dthree as (select distinct order_id, CASE
            WHEN selectedroute = 'hFRoute' THEN hfdistance
            WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
            WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
            ELSE NULL -- Handle other cases if needed
        END AS d3, selectedroute
        from
    (select * from (
    select *, row_number () over(partition by order_id order by updatedAt) rn from (
     select yyyymmdd,orderid as order_id, cast(hfdistance as double)*1000 as hfdistance,cast(pingdensity as double)*100 as pingdensity,cast(directionsresponsedistance as double) as directionsresponsedistance,
                 cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,selectedroute,updatedAt
                 from raw.captain_distance_response
                    WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                   
                ))
                where rn = 1
                 )),
                 
    merged_tbl as
    (
    
    
        select done.order_id,d2,done.orderdate,d3,d1,
        -- group_tc,
        fare_recalculated_reason,spd_fraud_flag,order_status,service_name,
        CASE
            WHEN d2 IS NOT NULL then d1-d2
            END AS fe_vs_nav,
            d1-(d3*1.00/1000.00) as Est_Nav,
            case WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00)
            END AS nav_actual
        from done
        left join dthree
        on done.order_id=dthree.order_id
        left join distance2
        on done.order_id=distance2.order_id
    
    
    
    )
    -- select * from merged_tbl
    -- where orderdate is null
    
    
    select service_name,
    orderdate,
    -- ,
    count(distinct order_id) as gross_orders,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end) as net_orders,
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end)*100.00)/(count (distinct order_id)) as "G2N",
    
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as fe_nav_bucket_within_500m_percent,
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as nav_actual_bucket_within_500m_percent,
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end)*100.00)/(count( distinct order_id)) as route_diff_percent,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
    
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='b. Less_than_500m' then order_id end) as fe_nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='c. Greater_than_500m' then order_id end) as fe_nav_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end) as nav_actual_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='b. Less_than_500m' then order_id end) as nav_actual_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='c. Greater_than_500m' then order_id end) as nav_actual_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='a. Within_500_m' then order_id end) as Est_Nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='b. Less_than_500m' then order_id end) as Est_Nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='c. Greater_than_500m' then order_id end) as Est_Nav_bucket_grt_500m
    
    
    
    from(
            select *,case when d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket,
            case when  d2 is not NULL and nav_actual>=-0.5 and nav_actual<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and nav_actual<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and nav_actual>0.5 then 'c. Greater_than_500m' end as nav_actual_bucket,
            case when Est_Nav>=-0.5 and Est_Nav<=0.5 then 'a. Within_500_m'
            when Est_Nav<-0.5 then 'b. Less_than_500m'
            when Est_Nav>0.5 then 'c. Greater_than_500m' end as Est_Nav_bucket
            from merged_tbl
            group by 1,2,3,4,5,6,7,8,9,10,11,12,13
    )
    group by 1,2
    order by 2
"""

In [11]:
route_check_df_1=pd.read_sql(route_check1,connection3)

In [16]:
route_check_df_1['fe_nav_bucket_within_500m_percent']=(route_check_df_1['fe_nav_bucket_within_500m']*100.00)/route_check_df_1['net_orders']
route_check_df_1['nav_actual_bucket_within_500m_percent']=(route_check_df_1['nav_actual_bucket_within_500m']*100.00)/route_check_df_1['net_orders']
route_check_df_1['route_diff_percent']=(route_check_df_1['route_diff_orders']*100.00)/route_check_df_1['net_orders']

In [17]:
33171/156004

0.21262916335478577

In [29]:
route_check_df_1

,service_name,orderdate,gross_orders,net_orders,G2N,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,route_diff_percent,route_diff_orders,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
0,Auto,2024-04-25,417522,156004,37.36,85.740109,74.014128,21.262916,33171,133758,13003,16309,115465,20080,27549,118039,14487,23464
1,Auto,2024-04-26,264968,114633,43.26,86.653930,74.525660,20.901486,23960,99334,9104,9723,85431,13410,19314,87990,10698,15939
2,Auto,2024-04-27,286963,141584,49.34,86.956153,75.344672,20.469121,28981,123116,10820,11212,106676,15559,22922,109254,13021,19302
3,Auto,2024-04-28,263751,129513,49.10,86.165868,74.628030,21.260414,27535,111596,10803,8913,96653,13192,21505,98998,12564,17941
4,Auto,2024-04-29,327712,146973,44.85,87.042518,75.434944,19.222578,28252,127929,11892,8585,110869,13510,24170,114865,13031,19063
5,Auto,2024-04-30,335270,144549,43.11,87.393202,75.647012,19.178964,27723,126326,12193,7394,109347,13003,23707,112682,13378,18473
6,Auto,2024-05-01,264856,137119,51.77,87.211838,76.037602,19.916277,27309,119584,12206,6503,104262,11719,22441,106915,12803,17393
7,Auto,2024-05-02,408310,152303,37.30,86.858434,75.062868,20.271433,30874,132288,12263,9342,114323,13077,26631,116853,12895,22529
8,Auto,2024-05-03,348866,152654,43.76,89.662243,75.234845,20.424620,31179,136873,9318,7518,114849,11421,27453,116473,10865,25184
9,Auto,2024-05-04,335079,157533,47.01,89.770397,75.501006,20.833095,32819,141418,9652,7989,118939,12456,27749,120277,11845,25386


In [26]:
city_name = 'Jaipur'
start_date='2024-02-09'
end_date='2024-02-14'
service_name='Auto'
route_check2=f"""WITH service_mapping AS ( 
        SELECT 
            service_detail_id,
            service_level as service_name,
            service_category,
            service_id,
            city_display_name as city,
            city_id
        FROM 
            datasets.service_mapping
        WHERE 
            city_display_name='{city_name}'
            and service_level='{service_name}'
    ),
    
    fe_tbl as (
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1
            FROM
                pricing.fare_estimates_enriched 
            WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
               
    ),
    orderlogs as 
    (SELECT *,u.id_array as fare_estimate_id
    FROM 
    ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
             date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
             epoch,estimate_id,estimate_ids,ride_time as t3,
             fare_recalculated_reason, all_paths,
             spd_fraud_flag,order_status,
             service_obj_service_name as service_name,
             fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
             cast(json_parse(estimate_ids) as array<varchar>) AS id_array
      FROM orders.order_logs_snapshot
           WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)) as t
    CROSS JOIN UNNEST(t.id_array) AS u(id_array)),
    immutable AS (
        SELECT 
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
            order_id,
            cast (event_time as double) event_time
        FROM 
            orders.order_logs_immutable 
           WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_type = 'started'
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
    ),
    
    nav_metrics AS (
        SELECT 
            event_props_order_id,
            cast  (event_props_epoch as double) event_props_epoch,
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
            try(cast(event_props_distance_in_meters as double) )/ 1000 d2
        FROM
            clevertap.captain_inappnavmetrics_immutable
             WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_props_order_status = 'started'
    ),
    
    distance2 AS (
        SELECT 
            orderdate,
            order_id,
            d2
        FROM 
            ( 
            select 
                imu.orderdate,
                imu.order_id,
                date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                "d2"
            from 
                immutable imu 
            join 
                nav_metrics nav 
                ON imu.order_id = nav.event_props_order_id
            )
            --and event_props_epoch >= event_time )
        WHERE abs(minDiff) <= 1
    ),
    done as  (select 
               distinct order_id,d1,t1,t3,epoch,fare_recalculated_reason,service_name,rn,api_context, (fe_amount-order_amount) as amt_diff, all_paths,fid,
            spd_fraud_flag,order_status,orderdate
           
               from 
              (SELECT *,row_number () over(partition by order_id order by fe) rn from 
              (SELECT  *
               from orderlogs join fe_tbl
               on fare_estimate_id = fid 
               and service_detail_id = service_id_fare)
               )
               where rn =1
     ),
    dthree as (select distinct order_id, CASE
            WHEN selectedroute = 'hFRoute' THEN hfdistance
            WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
            WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
            ELSE NULL -- Handle other cases if needed
        END AS d3, selectedroute
        from
    (select * from (
    select *, row_number () over(partition by order_id order by updatedAt) rn from (
     select yyyymmdd,orderid as order_id, cast(hfdistance as double)*1000 as hfdistance,cast(pingdensity as double)*100 as pingdensity,cast(directionsresponsedistance as double) as directionsresponsedistance,
                 cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,selectedroute,updatedAt
                 from raw.captain_distance_response
                    WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                   
                ))
                where rn = 1
                 )),
                 
    merged_tbl as
    (
    
    
        select done.order_id,d2,done.orderdate,d3,d1,
        -- group_tc,
        fare_recalculated_reason,spd_fraud_flag,order_status,service_name,
        CASE
            WHEN d2 IS NOT NULL then d1-d2
            END AS fe_vs_nav,
            d1-(d3*1.00/1000.00) as Est_Nav,
            case WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00)
            END AS nav_actual
        from done
        left join dthree
        on done.order_id=dthree.order_id
        left join distance2
        on done.order_id=distance2.order_id
    
    
    
    )
    -- select * from merged_tbl
    -- where orderdate is null
    
    
    select service_name,
    orderdate,
    -- ,
    count(distinct order_id) as gross_orders,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end) as net_orders,
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end)*100.00)/(count (distinct order_id)) as "G2N",
    
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as fe_nav_bucket_within_500m_percent,
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as nav_actual_bucket_within_500m_percent,
    (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end)*100.00)/(count( distinct order_id)) as route_diff_percent,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
    
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='b. Less_than_500m' then order_id end) as fe_nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='c. Greater_than_500m' then order_id end) as fe_nav_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end) as nav_actual_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='b. Less_than_500m' then order_id end) as nav_actual_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='c. Greater_than_500m' then order_id end) as nav_actual_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='a. Within_500_m' then order_id end) as Est_Nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='b. Less_than_500m' then order_id end) as Est_Nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='c. Greater_than_500m' then order_id end) as Est_Nav_bucket_grt_500m
    
    
    
    from(
            select *,case when d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket,
            case when  d2 is not NULL and nav_actual>=-0.5 and nav_actual<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and nav_actual<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and nav_actual>0.5 then 'c. Greater_than_500m' end as nav_actual_bucket,
            case when Est_Nav>=-0.5 and Est_Nav<=0.5 then 'a. Within_500_m'
            when Est_Nav<-0.5 then 'b. Less_than_500m'
            when Est_Nav>0.5 then 'c. Greater_than_500m' end as Est_Nav_bucket
            from merged_tbl
            group by 1,2,3,4,5,6,7,8,9,10,11,12,13
    )
    group by 1,2
    order by 2
"""
route_check_df_2=pd.read_sql(route_check2,connection3)


In [27]:
route_check_df_2['fe_nav_bucket_within_500m_percent']=(route_check_df_2['fe_nav_bucket_within_500m']*100.00)/route_check_df_2['net_orders']
route_check_df_2['nav_actual_bucket_within_500m_percent']=(route_check_df_2['nav_actual_bucket_within_500m']*100.00)/route_check_df_2['net_orders']
route_check_df_2['route_diff_percent']=(route_check_df_2['route_diff_orders']*100.00)/route_check_df_2['net_orders']

In [28]:
route_check_df_2['Est_Nav_bucket_less_500m_Percent']=(route_check_df_2['Est_Nav_bucket_less_500m']*100.00)/route_check_df_2['net_orders']

,service_name,orderdate,gross_orders,net_orders,G2N,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,route_diff_percent,route_diff_orders,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
0,Auto,2024-02-09,16647,8241,49.50,82.004611,69.858027,29.959956,2469,6758,772,712,5757,1020,1472,5737,1030,1473
1,Auto,2024-02-10,18560,8546,46.05,75.438802,64.696934,31.383103,2682,6447,1125,639,5529,1065,1609,5821,1294,1430
2,Auto,2024-02-11,20926,7919,37.84,76.019699,63.164541,33.526960,2655,6020,986,600,5002,1035,1563,5216,1242,1460
3,Auto,2024-02-12,24025,11601,48.29,74.622877,62.916990,33.212654,3853,8657,1405,1061,7299,1507,2262,7699,1698,2202
4,Auto,2024-02-13,22846,11521,50.43,75.818071,63.275757,33.009287,3803,8735,1405,943,7290,1474,2279,7658,1736,2124
5,Auto,2024-02-14,32696,12367,37.82,71.512897,61.890515,33.912833,4194,8844,1954,1402,7654,1851,2620,8106,1971,2228


In [30]:
route_check_df['fe_nav_bucket_within_500m_percent']=(route_check_df['fe_nav_bucket_within_500m']*100.00)/route_check_df['net_orders']
route_check_df['nav_actual_bucket_within_500m_percent']=(route_check_df['nav_actual_bucket_within_500m']*100.00)/route_check_df['net_orders']
route_check_df['route_diff_percent']=(route_check_df['route_diff_orders']*100.00)/route_check_df['net_orders']

In [39]:
route_check_df['Est_Nav_bucket_less_500m_Percent']=(route_check_df['Est_Nav_bucket_less_500m']*100.00)/route_check_df['net_orders']

In [41]:
route_check_df['Est_Nav_bucket_grt_500m_Percent']=(route_check_df['Est_Nav_bucket_grt_500m']*100.00)/route_check_df['net_orders']

In [42]:
route_check_df

,service_name,orderdate,gross_orders,net_orders,G2N,route_diff_orders,fe_nav_bucket_within_500m_percent,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,nav_actual_bucket_within_500m_percent,route_diff_percent,Est_Nav_bucket_less_500m_Percent,Est_Nav_bucket_grt_500m_Percent
0,CabEconomy,2024-04-25,52496,23744,45.23,7560,80.925708,19215,2347,3244,15904,5022,3756,15591,4426,3725,66.981132,31.839623,18.640499,15.688174
1,CabEconomy,2024-04-26,43344,19073,44.00,5953,82.215698,15681,1809,2112,12858,3771,2906,12611,3547,2914,67.414670,31.211660,18.596970,15.278142
2,CabEconomy,2024-04-27,47075,22144,47.04,6952,82.667991,18306,1933,2406,14940,4451,3169,14723,4196,3223,67.467486,31.394509,18.948699,14.554733
3,CabEconomy,2024-04-28,53933,21930,40.66,6989,81.171911,17801,2150,2265,14681,4190,3265,14384,4182,3362,66.944824,31.869585,19.069767,15.330597
4,CabEconomy,2024-04-29,44584,21774,48.84,6489,82.887848,18048,2093,1879,14815,3969,3183,14670,4059,3044,68.039864,29.801598,18.641499,13.979976
5,CabEconomy,2024-04-30,42184,22176,52.57,6599,82.183442,18225,2446,1693,15216,3646,3493,15073,4028,3073,68.614719,29.757395,18.163781,13.857323
6,CabEconomy,2024-05-01,44411,22847,51.44,6895,82.461592,18840,2539,1629,15696,3824,3447,15392,4394,3060,68.700486,30.179017,19.232284,13.393443
7,CabEconomy,2024-05-02,51897,23737,45.74,7200,82.339807,19545,2382,1999,16361,3808,3706,16036,4117,3581,68.926149,30.332392,17.344231,15.086152
8,CabEconomy,2024-05-03,46317,23869,51.53,6644,86.283464,20595,1885,1603,16948,3244,3865,16774,3347,3737,71.004231,27.835268,14.022372,15.656291
9,CabEconomy,2024-05-04,54846,25387,46.29,7556,85.843148,21793,2030,1736,17678,3583,4268,17438,3769,4173,69.634065,29.763265,14.846181,16.437547


In [33]:
route_check_df_1

,service_name,orderdate,gross_orders,net_orders,G2N,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,route_diff_percent,route_diff_orders,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
0,Auto,2024-04-25,417522,156004,37.36,85.740109,74.014128,21.262916,33171,133758,13003,16309,115465,20080,27549,118039,14487,23464
1,Auto,2024-04-26,264968,114633,43.26,86.653930,74.525660,20.901486,23960,99334,9104,9723,85431,13410,19314,87990,10698,15939
2,Auto,2024-04-27,286963,141584,49.34,86.956153,75.344672,20.469121,28981,123116,10820,11212,106676,15559,22922,109254,13021,19302
3,Auto,2024-04-28,263751,129513,49.10,86.165868,74.628030,21.260414,27535,111596,10803,8913,96653,13192,21505,98998,12564,17941
4,Auto,2024-04-29,327712,146973,44.85,87.042518,75.434944,19.222578,28252,127929,11892,8585,110869,13510,24170,114865,13031,19063
5,Auto,2024-04-30,335270,144549,43.11,87.393202,75.647012,19.178964,27723,126326,12193,7394,109347,13003,23707,112682,13378,18473
6,Auto,2024-05-01,264856,137119,51.77,87.211838,76.037602,19.916277,27309,119584,12206,6503,104262,11719,22441,106915,12803,17393
7,Auto,2024-05-02,408310,152303,37.30,86.858434,75.062868,20.271433,30874,132288,12263,9342,114323,13077,26631,116853,12895,22529
8,Auto,2024-05-03,348866,152654,43.76,89.662243,75.234845,20.424620,31179,136873,9318,7518,114849,11421,27453,116473,10865,25184
9,Auto,2024-05-04,335079,157533,47.01,89.770397,75.501006,20.833095,32819,141418,9652,7989,118939,12456,27749,120277,11845,25386


In [36]:
pd.concat([route_check_df,route_check_df_1]).to_csv('check_orders.csv')

In [130]:
def route_cluster_level(start_date,end_date,service_name,city):
    route_level_longest=f"""with orderlogs as 
    (SELECT *,u.id_array as fare_estimate_id
    FROM 
    ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
             date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
             epoch,estimate_id,estimate_ids,ride_time as t3,
             pickup_cluster || ' to ' || drop_cluster cluster_route,
             city_name,
             service_obj_service_name as service_name,
             fare_recalculated_reason, all_paths,
             fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
             cast(json_parse(estimate_ids) as array<varchar>) AS id_array
      FROM orders.order_logs_snapshot
         WHERE
                  yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
        and order_status = 'dropped'
        and (spd_fraud_flag = false or spd_fraud_flag is null) 
        AND service_obj_service_name = '{service_name}' 
        and captain_app_version >= '5.3.22'
       and city_name = '{city}') as t
    CROSS JOIN UNNEST(t.id_array) AS u(id_array))
    
    ,fe_tbl as (
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1
            FROM
                pricing.fare_estimates_enriched 
            WHERE
                  yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
               
    )
    
    ,done as  (select 
               distinct order_id,d1,t1,t3,epoch,orderdate,
               CASE WHEN orderdate <= ('2024-05-01') THEN 'pre' else 'post' end as new_date_type,
               fare_recalculated_reason,rn,api_context, (fe_amount-order_amount) as amt_diff, all_paths,fid,cluster_route,service_name,city_name
               from 
              (SELECT *,row_number () over(partition by order_id order by fe) rn from 
              (SELECT  *
               from orderlogs join fe_tbl
               on fare_estimate_id = fid 
               and service_detail_id = service_id_fare)
               )
               where rn =1
     )
     
    ,dthree as (select distinct order_id, CASE
            WHEN selectedroute = 'hFRoute' THEN hfdistance
            WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
            WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
            ELSE NULL -- Handle other cases if needed
        END AS d3, selectedroute
        from
    (select * from (
    select *, row_number () over(partition by order_id order by updatedAt) rn from (
     select yyyymmdd,orderid as order_id, cast(hfdistance as double)*1000 as hfdistance,cast(pingdensity as double)*100 as pingdensity,cast(directionsresponsedistance as double) as directionsresponsedistance,
                 cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,selectedroute,updatedAt
                 from raw.captain_distance_response
                  WHERE
                  yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                   
                ))
                where rn = 1
                 )) 
                 
    ,gtable as (  
    select order_id, d1,t1,t3,d3,orderdate,routes_cnt,city_name,cluster_route,service_name,
    new_date_type,
           ARRAY_SORT(ARRAY[distance1, distance2, distance3])[1] AS dist1,
        ARRAY_SORT(ARRAY[distance1, distance2, distance3])[2] AS dist2,
        ARRAY_SORT(ARRAY[distance1, distance2, distance3])[3] AS dist3,
           ARRAY_SORT(ARRAY[time1, time2, time3])[1] AS time1,
           ARRAY_SORT(ARRAY[time1, time2, time3])[2] AS time2,
           ARRAY_SORT(ARRAY[time1, time2, time3])[3] AS time3
        from
    (select done.order_id,t1,t3,orderdate, d1,json_array_length(all_paths) AS routes_cnt,
    new_date_type,
        cast(json_extract_scalar(all_paths, '$[0].distance') as double) /1000 AS distance1,
        cast(json_extract_scalar(all_paths, '$[0].time') as double)/60 AS time1,
          cast(json_extract_scalar(all_paths, '$[1].distance') as double) /1000 AS distance2,
           cast(json_extract_scalar(all_paths, '$[1].time') as double) /60 AS time2,
           cast(json_extract_scalar(all_paths, '$[2].distance') as double) /1000 AS distance3,
           cast(json_extract_scalar(all_paths, '$[2].time') as double) /60 AS time3,
           d3/1000 as d3,city_name,cluster_route,service_name
           FROM
            done
        join dthree on done.order_id = dthree.order_id 
         ))  
      
    
    ,r1 as (select
        ord, d1,d3,dist1,orderdate,city_name,cluster_route,routes_cnt,service_name,new_date_type,(CASE
        WHEN dist2 IS NOT NULL THEN
            CASE
                WHEN dist1 >= dist2 THEN dist1
                ELSE dist2
            END
        ELSE
            dist1
    END) AS dist2,
    (CASE
        WHEN dist3 IS NOT NULL THEN
            CASE
                WHEN dist3 >= COALESCE(dist2, dist3) AND dist3 >= COALESCE(dist1, dist3) THEN dist3
                WHEN dist2 >= COALESCE(dist1, dist2) THEN dist2
                ELSE dist1
            END
        ELSE
            CASE
                WHEN dist1 >= COALESCE(dist2, dist1) THEN dist1
                ELSE dist2
            END
    END) AS dist3
    
    from (
    select DISTINCT order_id AS ord,orderdate,d1,d3,routes_cnt,new_date_type,dist1,dist2,dist3,t1, t3,time1,time2,time3,city_name,cluster_route,service_name from gtable
    
    ))
    
    
    ,r2 AS (
        SELECT 
             ord,orderdate,city_name,cluster_route,service_name,routes_cnt,new_date_type,
            (d1-d3) as Est_nav
        FROM 
            r1
             
    )
    
     SELECT city_name,cluster_route,
     service_name,new_date_type,MIN(orderdate) || MAX(orderdate),
        count(distinct ord) as orders_count,
        (count(distinct case when est_nav>0.5 then ord END)*100.000) / count(distinct ord) AS "D1-D3>500m % (Negative Fare Shock)",
        (count(distinct case when est_nav<-0.5 then ord END)*100.000) / count(distinct ord) AS "D1-D3<-500m% (Positive Fare Shock)",
        (count(distinct case when est_nav>=-0.5 and  Est_nav<=0.5 then ord END)*100.000) / count(distinct ord) AS "No Fare Shock",
        count(CASE WHEN Est_nav <= -20  THEN ord END)*1.000 / count(ord) AS "< -20",
        count(CASE WHEN Est_nav <= -15 AND Est_nav > -20 THEN ord END)*1.000 / count(ord) AS "-20 to -15",
        count(CASE WHEN Est_nav <= -10 AND Est_nav > -15 THEN ord END)*1.000 / count(ord) AS "-15 to -10",
        count(CASE WHEN Est_nav <= -5 AND Est_nav > -10 THEN ord END)*1.000 / count(ord) AS "-10 to -5",
        count(CASE WHEN Est_nav <= -2 AND Est_nav > -5 THEN ord END)*1.000 / count(ord) AS "-5 to -2",
        count(CASE WHEN Est_nav <= -1 AND Est_nav > -2 THEN ord END)*1.000 / count(ord) AS "-2 to -1",
        count(CASE WHEN Est_nav <= -0.5 AND Est_nav > -1 THEN ord END)*1.000 / count(ord) AS "-1 to -0.5",
        count(CASE WHEN Est_nav <= -0.2 AND Est_nav > -0.5 THEN ord END)*1.000 / count(ord) AS "-0.5 to -0.2",
        count(CASE WHEN Est_nav < 0 AND Est_nav > -0.2 THEN ord END)*1.000 / count(ord) AS "-0.2 to 0",
        count(CASE WHEN Est_nav = 0 THEN ord END)*1.000 / count(ord) AS "0",
        count(CASE WHEN Est_nav <= 0.2 AND Est_nav > 0 THEN ord END)*1.000 / count(ord) AS "0 to 0.2",
        count(CASE WHEN Est_nav <= 0.5 AND Est_nav > 0.2 THEN ord END)*1.000 / count(ord) AS "0.2 to 0.5",
        count(CASE WHEN Est_nav <= 1 AND Est_nav > 0.5 THEN ord END)*1.000 / count(ord) AS "0.5 to 1",
        count(CASE WHEN Est_nav <= 2 AND Est_nav > 1 THEN ord END)*1.000 / count(ord) AS "1 to 2",
        count(CASE WHEN Est_nav <= 5 AND Est_nav > 2 THEN ord END)*1.000 / count(ord) AS "2 to 5",
        count(CASE WHEN Est_nav <= 10 AND Est_nav > 5 THEN ord END)*1.000 / count(ord) AS "5 to 10",
        count(CASE WHEN Est_nav <= 15 AND Est_nav > 10 THEN ord END)*1.000 / count(ord) AS "10 to 15",
        count(CASE WHEN Est_nav <= 20 AND Est_nav > 15 THEN ord END)*1.000 / count(ord) AS "15 to 20",
        count(CASE WHEN  Est_nav > 20 THEN ord END)*1.000 / count(ord) AS ">20"
      FROM r2
      group by 1,2,3,4
     """
    d1=pd.read_sql(route_level_longest,connection4)
    return d1

In [131]:
start_date='20240425'
end_date='20240506'

In [132]:
total_df_second_longest1=pd.DataFrame()
city_name=['Bangalore','Delhi','Hyderabad','Chennai','Jaipur']
service_level=['Link','Auto','CabEconomy']
for i in city_name[0:1]:
    print(i)
    for j in service_level[1:]:
        print(j)
        df=route_cluster_level(start_date,end_date,j,i)
        df['distance_type']='second_longest'
        total_df_second_longest1=pd.concat([total_df_second_longest1,df])

Bangalore
Auto


DatabaseError: {'message': "line 20:21: mismatched input 'app'. Expecting: '%', '(', ')', '*', '+', ',', '-', '.', '/', 'AND', 'AS', 'AT', 'CROSS', 'EXCEPT', 'FETCH', 'FULL', 'GROUP', 'HAVING', 'INNER', 'INTERSECT', 'JOIN', 'LEFT', 'LIMIT', 'MATCH_RECOGNIZE', 'NATURAL', 'OFFSET', 'OR', 'ORDER', 'RIGHT', 'SELECT', 'TABLE', 'TABLESAMPLE', 'UNION', 'VALUES', 'WHERE', 'WINDOW', '[', '||', <EOF>, <identifier>, <predicate>", 'errorCode': 1, 'errorName': 'SYNTAX_ERROR', 'errorType': 'USER_ERROR', 'errorLocation': {'lineNumber': 20, 'columnNumber': 21}, 'failureInfo': {'type': 'io.trino.sql.parser.ParsingException', 'message': "line 20:21: mismatched input 'app'. Expecting: '%', '(', ')', '*', '+', ',', '-', '.', '/', 'AND', 'AS', 'AT', 'CROSS', 'EXCEPT', 'FETCH', 'FULL', 'GROUP', 'HAVING', 'INNER', 'INTERSECT', 'JOIN', 'LEFT', 'LIMIT', 'MATCH_RECOGNIZE', 'NATURAL', 'OFFSET', 'OR', 'ORDER', 'RIGHT', 'SELECT', 'TABLE', 'TABLESAMPLE', 'UNION', 'VALUES', 'WHERE', 'WINDOW', '[', '||', <EOF>, <iden

In [125]:
total_df_second_longest

,city_name,service_name,new_date_type,_col3,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
0,Bangalore,Auto,post,2024-05-022024-05-06,756629,15.962,7.606,76.433,0.000,0.000,0.000,0.002,0.011,0.024,0.038,0.061,0.144,0.001,0.398,0.160,0.089,0.045,0.020,0.004,0.001,0.000,0.000,second_longest
1,Bangalore,Auto,pre,2024-04-252024-05-01,970392,13.567,9.273,77.161,0.000,0.000,0.000,0.002,0.013,0.029,0.047,0.074,0.158,0.001,0.384,0.155,0.080,0.038,0.014,0.002,0.001,0.000,0.000,second_longest
0,Bangalore,CabEconomy,post,2024-05-022024-05-06,122884,15.883,14.990,69.127,0.001,0.001,0.001,0.007,0.031,0.048,0.061,0.080,0.152,0.001,0.319,0.139,0.075,0.044,0.026,0.007,0.003,0.002,0.002,second_longest
1,Bangalore,CabEconomy,pre,2024-04-252024-05-01,153704,14.578,18.763,66.659,0.000,0.001,0.002,0.010,0.041,0.059,0.074,0.090,0.150,0.001,0.293,0.133,0.073,0.043,0.020,0.005,0.002,0.001,0.001,second_longest


In [124]:
total_df_second_longest1.head(10)

,city_name,cluster_route,service_name,new_date_type,_col4,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
0,Bangalore,Singasandra Lake to Koramangala 3rd Block,Auto,pre,2024-04-252024-04-25,1,0.000,0.000,100.000,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,1.000,0.000,0.000,0.000,0.00,0.0,0.0,0.0,second_longest
1,Bangalore,Kathriguppe to Gattigere,Auto,pre,2024-04-252024-05-01,177,11.864,12.994,75.141,0.0,0.0,0.0,0.000,0.011,0.056,0.062,0.107,0.147,0.0,0.333,0.164,0.085,0.023,0.011,0.00,0.0,0.0,0.0,second_longest
2,Bangalore,Garvebhavipalya Lake to Majestic,Auto,pre,2024-04-252024-05-01,98,18.367,8.163,73.469,0.0,0.0,0.0,0.000,0.010,0.010,0.061,0.092,0.122,0.0,0.255,0.265,0.133,0.031,0.010,0.01,0.0,0.0,0.0,second_longest
3,Bangalore,Kathriguppe to Subramanyapura,Auto,pre,2024-04-252024-05-01,295,12.542,6.780,80.678,0.0,0.0,0.0,0.000,0.017,0.014,0.037,0.098,0.176,0.0,0.414,0.119,0.105,0.017,0.003,0.00,0.0,0.0,0.0,second_longest
4,Bangalore,Marathahalli Ring Road Junc to Munnekollal,Auto,pre,2024-04-252024-05-01,285,9.474,2.456,88.070,0.0,0.0,0.0,0.000,0.004,0.007,0.014,0.049,0.182,0.0,0.551,0.098,0.035,0.035,0.025,0.00,0.0,0.0,0.0,second_longest
5,Bangalore,Sampagni Rama Nagar to Jakkur,Auto,pre,2024-04-252024-04-30,15,13.333,0.000,86.667,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.133,0.133,0.0,0.400,0.200,0.067,0.067,0.000,0.00,0.0,0.0,0.0,second_longest
6,Bangalore,Koramangala 5th Block to Ejipura,Auto,pre,2024-04-252024-05-01,253,4.348,1.186,94.466,0.0,0.0,0.0,0.000,0.000,0.000,0.012,0.047,0.249,0.0,0.522,0.126,0.032,0.012,0.000,0.00,0.0,0.0,0.0,second_longest
7,Bangalore,Hongasandra to Kudlu Gate,Auto,post,2024-05-022024-05-06,67,5.970,4.478,89.552,0.0,0.0,0.0,0.000,0.015,0.000,0.030,0.045,0.164,0.0,0.567,0.119,0.015,0.045,0.000,0.00,0.0,0.0,0.0,second_longest
8,Bangalore,RTO Rajajinagar to Yelahanka New Town,Auto,pre,2024-04-252024-05-01,64,9.375,17.188,73.438,0.0,0.0,0.0,0.016,0.016,0.094,0.047,0.156,0.047,0.0,0.344,0.188,0.047,0.047,0.000,0.00,0.0,0.0,0.0,second_longest
9,Bangalore,Banashankri South to Banashankri,Auto,post,2024-05-022024-05-06,194,11.340,7.216,81.443,0.0,0.0,0.0,0.000,0.000,0.036,0.036,0.103,0.160,0.0,0.397,0.155,0.088,0.026,0.000,0.00,0.0,0.0,0.0,second_longest


In [119]:
total_df_second_longest.head(10)

,city_name,service_name,new_date_type,_col3,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
0,Bangalore,Auto,post,2024-05-022024-05-06,756629,15.962,7.606,76.433,0.000,0.000,0.000,0.002,0.011,0.024,0.038,0.061,0.144,0.001,0.398,0.160,0.089,0.045,0.020,0.004,0.001,0.000,0.000,second_longest
1,Bangalore,Auto,pre,2024-04-252024-05-01,970392,13.567,9.273,77.161,0.000,0.000,0.000,0.002,0.013,0.029,0.047,0.074,0.158,0.001,0.384,0.155,0.080,0.038,0.014,0.002,0.001,0.000,0.000,second_longest
0,Bangalore,CabEconomy,post,2024-05-022024-05-06,122884,15.883,14.990,69.127,0.001,0.001,0.001,0.007,0.031,0.048,0.061,0.080,0.152,0.001,0.319,0.139,0.075,0.044,0.026,0.007,0.003,0.002,0.002,second_longest
1,Bangalore,CabEconomy,pre,2024-04-252024-05-01,153704,14.578,18.763,66.659,0.000,0.001,0.002,0.010,0.041,0.059,0.074,0.090,0.150,0.001,0.293,0.133,0.073,0.043,0.020,0.005,0.002,0.001,0.001,second_longest


In [106]:
total_df_second_longest[total_df_second_longest['cluster_route']=='BLR_Indiranagar to BLR_Indiranagar']

,city_name,cluster_route,service_name,new_date_type,_col4,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
5095,Bangalore,BLR_Indiranagar to BLR_Indiranagar,Auto,post,2024-05-022024-05-06,2304,4.688,1.476,93.837,0.0,0.0,0.0,0.0,0.002,0.001,0.011,0.039,0.188,0.001,0.545,0.166,0.038,0.007,0.001,0.000,0.000,0.0,0.000,second_longest
61318,Bangalore,BLR_Indiranagar to BLR_Indiranagar,Auto,pre,2024-04-252024-05-01,3005,4.326,1.265,94.409,0.0,0.0,0.0,0.0,0.000,0.003,0.009,0.046,0.221,0.003,0.537,0.138,0.036,0.004,0.001,0.001,0.000,0.0,0.000,second_longest
14279,Bangalore,BLR_Indiranagar to BLR_Indiranagar,CabEconomy,pre,2024-04-252024-05-01,66,4.545,1.515,93.939,0.0,0.0,0.0,0.0,0.000,0.000,0.015,0.076,0.333,0.000,0.455,0.076,0.000,0.015,0.000,0.015,0.000,0.0,0.015,second_longest
35234,Bangalore,BLR_Indiranagar to BLR_Indiranagar,CabEconomy,post,2024-05-022024-05-06,71,8.451,5.634,85.915,0.0,0.0,0.0,0.0,0.000,0.014,0.042,0.113,0.141,0.000,0.521,0.085,0.042,0.000,0.014,0.014,0.014,0.0,0.000,second_longest


In [103]:
total_df_second_longest.sort_values(by='orders_count',ascending=False).head(10)

,city_name,cluster_route,service_name,new_date_type,_col4,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
68888,Bangalore,Electronic City to Electronic City,Auto,pre,2024-04-252024-05-01,7879,11.385,4.036,84.579,0.0,0.0,0.0,0.000,0.002,0.010,0.029,0.051,0.155,0.001,0.464,0.176,0.084,0.024,0.004,0.000,0.001,0.001,0.0,second_longest
11577,Bangalore,Electronic City to Electronic City,Auto,post,2024-05-022024-05-06,5979,13.899,3.295,82.806,0.0,0.0,0.0,0.000,0.002,0.008,0.022,0.040,0.145,0.000,0.466,0.176,0.102,0.031,0.004,0.000,0.000,0.001,0.0,second_longest
30625,Bangalore,Horamavu to Horamavu,Auto,pre,2024-04-252024-05-01,3459,6.736,3.758,89.506,0.0,0.0,0.0,0.000,0.005,0.012,0.021,0.047,0.171,0.000,0.546,0.130,0.055,0.010,0.002,0.000,0.001,0.000,0.0,second_longest
41024,Bangalore,Majestic to Majestic,Auto,pre,2024-04-252024-05-01,3370,14.303,6.558,79.139,0.0,0.0,0.0,0.001,0.004,0.019,0.041,0.062,0.144,0.001,0.341,0.245,0.104,0.035,0.002,0.001,0.001,0.000,0.0,second_longest
61318,Bangalore,BLR_Indiranagar to BLR_Indiranagar,Auto,pre,2024-04-252024-05-01,3005,4.326,1.265,94.409,0.0,0.0,0.0,0.000,0.000,0.003,0.009,0.046,0.221,0.003,0.537,0.138,0.036,0.004,0.001,0.001,0.000,0.000,0.0,second_longest
14357,Bangalore,RTO Rajajinagar to RTO Rajajinagar,Auto,pre,2024-04-252024-05-01,2720,9.926,3.750,86.324,0.0,0.0,0.0,0.000,0.002,0.008,0.028,0.085,0.205,0.001,0.409,0.164,0.079,0.019,0.001,0.001,0.000,0.000,0.0,second_longest
21104,Bangalore,Horamavu to Horamavu,Auto,post,2024-05-022024-05-06,2710,8.266,3.727,88.007,0.0,0.0,0.0,0.000,0.004,0.007,0.026,0.038,0.147,0.000,0.565,0.129,0.062,0.017,0.002,0.001,0.000,0.000,0.0,second_longest
1306,Bangalore,Electronic City Phase 2 to Electronic City,Auto,pre,2024-04-252024-05-01,2687,13.063,5.471,81.466,0.0,0.0,0.0,0.000,0.005,0.015,0.035,0.060,0.156,0.000,0.443,0.156,0.065,0.047,0.017,0.000,0.000,0.001,0.0,second_longest
40685,Bangalore,Majestic to Majestic,Auto,post,2024-05-022024-05-06,2666,17.442,5.064,77.494,0.0,0.0,0.0,0.001,0.003,0.015,0.031,0.058,0.132,0.000,0.333,0.251,0.122,0.046,0.003,0.002,0.001,0.000,0.0,second_longest
45568,Bangalore,Sarjapur Road to Sarjapur Road,Auto,pre,2024-04-252024-05-01,2418,9.098,3.433,87.469,0.0,0.0,0.0,0.001,0.002,0.013,0.018,0.029,0.130,0.000,0.547,0.168,0.065,0.020,0.004,0.001,0.000,0.000,0.0,second_longest


In [101]:
total_df_second_longest[total_df_second_longest['cluster_route']=='Tavarakere to Tavarakere']

,city_name,cluster_route,service_name,new_date_type,_col4,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
0,Bangalore,Tavarakere to Tavarakere,Auto,pre,2024-04-252024-05-01,2012,10.239,1.889,87.873,0.0,0.0,0.0,0.0,0.0,0.003,0.015,0.027,0.172,0.000,0.518,0.161,0.080,0.020,0.002,0.0,0.0,0.00,0.0,second_longest
76033,Bangalore,Tavarakere to Tavarakere,Auto,post,2024-05-022024-05-06,1834,8.724,1.145,90.131,0.0,0.0,0.0,0.0,0.0,0.006,0.005,0.025,0.146,0.001,0.539,0.191,0.059,0.027,0.001,0.0,0.0,0.00,0.0,second_longest
7264,Bangalore,Tavarakere to Tavarakere,CabEconomy,post,2024-05-022024-05-06,20,10.000,5.000,85.000,0.0,0.0,0.0,0.0,0.0,0.000,0.050,0.150,0.250,0.000,0.350,0.100,0.050,0.000,0.000,0.0,0.0,0.05,0.0,second_longest
50637,Bangalore,Tavarakere to Tavarakere,CabEconomy,pre,2024-04-272024-05-01,11,0.000,0.000,100.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.091,0.182,0.000,0.636,0.091,0.000,0.000,0.000,0.0,0.0,0.00,0.0,second_longest


In [100]:
total_df_second_longest.head(10)

,city_name,cluster_route,service_name,new_date_type,_col4,orders_count,D1-D3>500m % (Negative Fare Shock),D1-D3<-500m% (Positive Fare Shock),No Fare Shock,< -20,-20 to -15,-15 to -10,-10 to -5,-5 to -2,-2 to -1,-1 to -0.5,-0.5 to -0.2,-0.2 to 0,0,0 to 0.2,0.2 to 0.5,0.5 to 1,1 to 2,2 to 5,5 to 10,10 to 15,15 to 20,>20,distance_type
0,Bangalore,Tavarakere to Tavarakere,Auto,pre,2024-04-252024-05-01,2012,10.239,1.889,87.873,0.0,0.0,0.0,0.000,0.000,0.003,0.015,0.027,0.172,0.000,0.518,0.161,0.080,0.020,0.002,0.0,0.0,0.0,0.0,second_longest
1,Bangalore,Chanasandra to Kadugodi Forest,Auto,pre,2024-04-252024-05-01,455,1.319,1.758,96.923,0.0,0.0,0.0,0.000,0.000,0.002,0.015,0.057,0.198,0.000,0.598,0.116,0.011,0.002,0.000,0.0,0.0,0.0,0.0,second_longest
2,Bangalore,Metro - MG Road to RTO Rajajinagar,Auto,pre,2024-04-252024-05-01,33,18.182,6.061,75.758,0.0,0.0,0.0,0.000,0.000,0.030,0.030,0.061,0.091,0.000,0.485,0.121,0.121,0.030,0.030,0.0,0.0,0.0,0.0,second_longest
3,Bangalore,Agara Lake to Nagarbhavi,Auto,post,2024-05-052024-05-05,1,0.000,0.000,100.000,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.0,second_longest
4,Bangalore,Peenya 2nd Stage to Banashankri North,Auto,post,2024-05-022024-05-06,40,20.000,12.500,67.500,0.0,0.0,0.0,0.000,0.000,0.050,0.075,0.075,0.200,0.025,0.300,0.075,0.075,0.075,0.050,0.0,0.0,0.0,0.0,second_longest
5,Bangalore,Peenya to Dasarahalli,Auto,pre,2024-04-252024-05-01,321,19.938,4.673,75.389,0.0,0.0,0.0,0.003,0.006,0.012,0.025,0.062,0.218,0.000,0.333,0.140,0.109,0.081,0.009,0.0,0.0,0.0,0.0,second_longest
6,Bangalore,Gottigere to Hsr,Auto,pre,2024-04-252024-05-01,28,7.143,10.714,82.143,0.0,0.0,0.0,0.000,0.000,0.036,0.071,0.036,0.286,0.000,0.357,0.143,0.071,0.000,0.000,0.0,0.0,0.0,0.0,second_longest
7,Bangalore,Hulimavu to Arekere Lake,Auto,post,2024-05-022024-05-06,40,5.000,0.000,95.000,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.075,0.200,0.000,0.525,0.150,0.050,0.000,0.000,0.0,0.0,0.0,0.0,second_longest
8,Bangalore,BLR_Ashok Nagar to ShivajiNagar - Bangalore,Auto,post,2024-05-022024-05-06,65,10.769,0.000,89.231,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.015,0.108,0.000,0.615,0.154,0.046,0.062,0.000,0.0,0.0,0.0,0.0,second_longest
9,Bangalore,Arekere Extension to Bannerghatta Road - Hulimavu,Auto,pre,2024-04-252024-05-01,133,5.263,3.008,91.729,0.0,0.0,0.0,0.000,0.000,0.015,0.015,0.015,0.120,0.000,0.602,0.180,0.045,0.008,0.000,0.0,0.0,0.0,0.0,second_longest


In [157]:
city_name = 'Bangalore'
start_date='2024-04-25'
end_date='2024-05-05'
service_name='CabEconomy'

In [158]:
check_app=f"""WITH service_mapping AS ( 
            SELECT 
                service_detail_id,
                service_level as service_name,
                service_category,
                service_id,
                city_display_name as city,
                city_id
            FROM 
                datasets.service_mapping
            WHERE 
                city_display_name='{city_name}'
                and service_level='{service_name}'
        ),
        
        fe_tbl as (
                SELECT
                    fare_estimate_id fid,
                    api_context,final_amount as final_amount_fare,
                    date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                    cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                    epoch fe,ride_distance as d1, estimated_ride_time as t1
                FROM
                    pricing.fare_estimates_enriched 
                WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d')
                   
        ),
        orderlogs as 
        (SELECT *,u.id_array as fare_estimate_id
        FROM 
        ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
                 date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                 epoch,estimate_id,estimate_ids,ride_time as t3,
                 fare_recalculated_reason, all_paths,
                 
                 spd_fraud_flag,order_status,
                 service_obj_service_name as service_name,
                 fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
                 cast(json_parse(estimate_ids) as array<varchar>) AS id_array
          FROM orders.order_logs_snapshot
                WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d')
                AND city_name IN (SELECT city FROM service_mapping)
                AND service_obj_service_name IN (SELECT service_name FROM service_mapping)) as t
        CROSS JOIN UNNEST(t.id_array) AS u(id_array)),
        immutable AS (
            SELECT 
                DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                order_id,
                cast (event_time as double) event_time
            FROM 
                orders.order_logs_immutable 
                 WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d')
                AND event_type = 'started'
                AND city_name IN (SELECT city FROM service_mapping)
                AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
        ),
        
        nav_metrics AS 
        (
            SELECT 
                event_props_order_id,device_info_app_version as app_version_type,
                event_props_user_id as user_id,
                -- case when device_info_app_version>='5.3.22' then 'Above 5.3.22' else 'Below 5.3.22' end as app_version_type,
                cast  (event_props_epoch as double) event_props_epoch,
                DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                try(cast(event_props_distance_in_meters as double) )/ 1000 d2
            FROM
                clevertap.captain_inappnavmetrics_immutable
                 WHERE 
                 yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_props_order_status = 'started'
        )
       select app_version_type,orderdate,(count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end)*100.00)/count(distinct order_id) from (
        select o.order_id,app_version_type,o.orderdate,event_props_order_id,user_id,order_status,spd_fraud_flag
        from orderlogs o left join nav_metrics n on o.order_id=n.event_props_order_id or o.captain_id=n.user_id
        )
        group by 1,2"""

In [159]:
check_app_df=pd.read_sql(check_app,connection3)

KeyboardInterrupt: 